# 📡 AI-Based RF Fingerprinting
**Goal:** Identify wireless devices based on unique RF (Radio Frequency) signatures.

**Applications:**
- Wireless security
- Device authentication

**Model:** CNN Classifier

**Pipeline Overview:**
1. Simulate / Load RF signal dataset
2. Extract I/Q samples and convert to spectrograms
3. Build & train CNN classifier
4. Evaluate performance
5. Visualize RF fingerprints

## 📦 Step 1: Install & Import Libraries

In [ ]:
# Install required libraries
!pip install numpy scipy matplotlib scikit-learn torch torchvision seaborn tqdm -q

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import LabelEncoder
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✅ Using device: {device}')

## 📡 Step 2: Simulate RF Signal Dataset (I/Q Samples)

Each device has a **unique RF fingerprint** due to hardware imperfections:
- Carrier Frequency Offset (CFO)
- Phase noise
- I/Q imbalance
- Nonlinearity

In [ ]:
np.random.seed(42)

# ─── Configuration ───────────────────────────────────────────
NUM_DEVICES      = 8          # Number of wireless devices
SAMPLES_PER_DEV  = 500        # Samples per device
SIGNAL_LENGTH    = 1024       # I/Q sample length
FS               = 1e6        # Sample rate (1 MHz)
SNR_DB           = 20         # Signal-to-noise ratio (dB)

def generate_rf_signal(device_id, n_samples=SIGNAL_LENGTH, fs=FS, snr_db=SNR_DB):
    """
    Simulate RF I/Q signal with device-specific hardware impairments.
    Each device has a unique CFO, phase noise level, and IQ imbalance.
    """
    t = np.arange(n_samples) / fs

    # ── Base carrier signal ──────────────────────────────────
    fc = 2.4e6  # Base carrier frequency (2.4 MHz)

    # ── Device-unique hardware impairments ───────────────────
    cfo        = (device_id + 1) * 0.5e3 + np.random.randn() * 100   # CFO (Hz)
    phase_noise_std = 0.01 * (device_id + 1)                          # Phase noise
    iq_imbalance    = 0.02 * (device_id + 1)                          # I/Q imbalance
    dc_offset_i     = 0.005 * (device_id + 1)                         # DC offset I
    dc_offset_q     = 0.003 * (device_id + 1)                         # DC offset Q

    # ── Generate base BPSK signal ────────────────────────────
    bits = np.random.randint(0, 2, n_samples)
    bpsk = 2 * bits - 1  # Map 0→-1, 1→+1

    # ── Apply impairments ────────────────────────────────────
    phase_noise = np.cumsum(np.random.randn(n_samples) * phase_noise_std)
    carrier_phase = 2 * np.pi * (fc + cfo) * t + phase_noise

    I = (1 + iq_imbalance) * bpsk * np.cos(carrier_phase) + dc_offset_i
    Q = (1 - iq_imbalance) * bpsk * np.sin(carrier_phase) + dc_offset_q

    # ── Add AWGN noise ───────────────────────────────────────
    snr_linear = 10 ** (snr_db / 10)
    signal_power = np.mean(I**2 + Q**2)
    noise_std = np.sqrt(signal_power / (2 * snr_linear))
    I += np.random.randn(n_samples) * noise_std
    Q += np.random.randn(n_samples) * noise_std

    return I, Q


def iq_to_spectrogram(I, Q, nperseg=64, noverlap=32):
    """Convert I/Q samples to 2D spectrogram for CNN input."""
    iq_complex = I + 1j * Q
    f, t_seg, Sxx = signal.spectrogram(
        iq_complex,
        fs=FS,
        window='hann',
        nperseg=nperseg,
        noverlap=noverlap,
        return_onesided=False
    )
    Sxx_db = 10 * np.log10(np.abs(Sxx) + 1e-12)
    # Resize to fixed shape 64x64
    from scipy.ndimage import zoom
    target = (64, 64)
    zoom_factors = (target[0] / Sxx_db.shape[0], target[1] / Sxx_db.shape[1])
    Sxx_resized = zoom(Sxx_db, zoom_factors)
    # Normalize to [0, 1]
    Sxx_norm = (Sxx_resized - Sxx_resized.min()) / (Sxx_resized.max() - Sxx_resized.min() + 1e-8)
    return Sxx_norm


# ─── Generate Dataset ────────────────────────────────────────
print('🔄 Generating RF signal dataset...')
X_list, y_list = [], []

for dev_id in range(NUM_DEVICES):
    for _ in tqdm(range(SAMPLES_PER_DEV), desc=f'Device {dev_id+1}/{NUM_DEVICES}', leave=False):
        I, Q = generate_rf_signal(dev_id)
        spec  = iq_to_spectrogram(I, Q)
        X_list.append(spec)
        y_list.append(dev_id)

X = np.array(X_list, dtype=np.float32)  # (N, 64, 64)
y = np.array(y_list, dtype=np.int64)

print(f'\n✅ Dataset shape: X={X.shape}, y={y.shape}')
print(f'   Classes: {NUM_DEVICES} devices, {SAMPLES_PER_DEV} samples each')

## 🔬 Step 3: Visualize RF Fingerprints

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle('RF Fingerprints — Spectrogram per Device', fontsize=15, fontweight='bold')

for dev_id in range(NUM_DEVICES):
    ax = axes[dev_id // 4][dev_id % 4]
    sample_idx = np.where(y == dev_id)[0][0]
    im = ax.imshow(X[sample_idx], aspect='auto', origin='lower',
                   cmap='inferno', interpolation='nearest')
    ax.set_title(f'Device {dev_id + 1}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Time bins')
    ax.set_ylabel('Freq bins')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig('rf_fingerprints.png', dpi=150, bbox_inches='tight')
plt.show()
print('📊 Spectrograms saved.')

## ✂️ Step 4: Train/Validation/Test Split & DataLoaders

In [ ]:
# Add channel dimension → (N, 1, 64, 64)
X = X[:, np.newaxis, :, :]

# 70% train | 15% val | 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42)

def make_loader(X_arr, y_arr, batch_size=64, shuffle=True):
    ds = TensorDataset(torch.from_numpy(X_arr), torch.from_numpy(y_arr))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

BATCH = 64
train_loader = make_loader(X_train, y_train, BATCH, shuffle=True)
val_loader   = make_loader(X_val,   y_val,   BATCH, shuffle=False)
test_loader  = make_loader(X_test,  y_test,  BATCH, shuffle=False)

print(f'Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

## 🧠 Step 5: Define CNN Model

In [ ]:
class RFFingerCNN(nn.Module):
    """
    CNN for RF Fingerprinting.
    Input:  (B, 1, 64, 64) spectrogram
    Output: (B, NUM_DEVICES) class logits
    """
    def __init__(self, num_classes=NUM_DEVICES):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),          # → 32×32
            nn.Dropout2d(0.25),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),          # → 16×16
            nn.Dropout2d(0.25),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),          # → 8×8
            nn.Dropout2d(0.25),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 8 * 8, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


model = RFFingerCNN(num_classes=NUM_DEVICES).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'✅ Model created  |  Trainable parameters: {total_params:,}')
print(model)

## 🏋️ Step 6: Train the CNN

In [ ]:
EPOCHS    = 30
LR        = 1e-3

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

best_val_acc = 0.0
best_model_path = 'best_rf_cnn.pth'

for epoch in range(1, EPOCHS + 1):
    # ── Training ─────────────────────────────────────────────
    model.train()
    train_loss, correct, total = 0.0, 0, 0
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(Xb)
        loss   = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * Xb.size(0)
        correct    += (logits.argmax(1) == yb).sum().item()
        total      += Xb.size(0)
    scheduler.step()
    tr_loss = train_loss / total
    tr_acc  = correct / total * 100

    # ── Validation ───────────────────────────────────────────
    model.eval()
    val_loss, vcorrect, vtotal = 0.0, 0, 0
    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            logits = model(Xb)
            loss   = criterion(logits, yb)
            val_loss  += loss.item() * Xb.size(0)
            vcorrect  += (logits.argmax(1) == yb).sum().item()
            vtotal    += Xb.size(0)
    vl_loss = val_loss / vtotal
    vl_acc  = vcorrect / vtotal * 100

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    # Save best model
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        torch.save(model.state_dict(), best_model_path)

    if epoch % 5 == 0 or epoch == 1:
        print(f'Epoch {epoch:3d}/{EPOCHS} | '
              f'Train Loss: {tr_loss:.4f}  Acc: {tr_acc:.2f}% | '
              f'Val Loss: {vl_loss:.4f}  Acc: {vl_acc:.2f}%')

print(f'\n🏆 Best Validation Accuracy: {best_val_acc:.2f}%')

## 📈 Step 7: Plot Training History

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('CNN Training History — RF Fingerprinting', fontsize=14, fontweight='bold')

epochs_range = range(1, EPOCHS + 1)

ax1.plot(epochs_range, history['train_loss'], 'b-o', markersize=3, label='Train Loss')
ax1.plot(epochs_range, history['val_loss'],   'r-o', markersize=3, label='Val Loss')
ax1.set_title('Loss'); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs_range, history['train_acc'], 'b-o', markersize=3, label='Train Acc')
ax2.plot(epochs_range, history['val_acc'],   'r-o', markersize=3, label='Val Acc')
ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy (%)')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150, bbox_inches='tight')
plt.show()

## 🧪 Step 8: Evaluate on Test Set

In [ ]:
# Load best model
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for Xb, yb in test_loader:
        Xb = Xb.to(device)
        preds = model(Xb).argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(yb.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)
test_acc   = (all_preds == all_labels).mean() * 100

print(f'\n✅ Test Accuracy: {test_acc:.2f}%\n')
device_names = [f'Device {i+1}' for i in range(NUM_DEVICES)]
print(classification_report(all_labels, all_preds, target_names=device_names))

## 🔥 Step 9: Confusion Matrix

In [ ]:
cm = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('RF Fingerprinting — Confusion Matrix', fontsize=14, fontweight='bold')

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=device_names, yticklabels=device_names, ax=axes[0])
axes[0].set_title('Counts'); axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')

sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='RdYlGn',
            xticklabels=device_names, yticklabels=device_names,
            vmin=0, vmax=1, ax=axes[1])
axes[1].set_title('Normalized'); axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')

plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔐 Step 10: Device Authentication Demo

In [ ]:
import torch.nn.functional as F

def authenticate_device(claimed_device_id, confidence_threshold=0.85):
    """Authenticate a device using RF fingerprinting."""
    # Generate a test signal from the claimed device
    I, Q = generate_rf_signal(claimed_device_id)
    spec  = iq_to_spectrogram(I, Q)
    x     = torch.from_numpy(spec[np.newaxis, np.newaxis, :, :].astype(np.float32)).to(device)

    model.eval()
    with torch.no_grad():
        logits = model(x)
        probs  = F.softmax(logits, dim=1)[0].cpu().numpy()

    predicted_id   = probs.argmax()
    confidence     = probs[predicted_id]
    identity_match = (predicted_id == claimed_device_id)
    auth_pass      = identity_match and (confidence >= confidence_threshold)

    print(f'\n{'='*55}')
    print(f'  Claimed Device  : Device {claimed_device_id + 1}')
    print(f'  Predicted Device: Device {predicted_id + 1}')
    print(f'  Confidence      : {confidence:.4f} ({confidence*100:.2f}%)')
    print(f'  Threshold       : {confidence_threshold}')
    print(f'  Identity Match  : {"✅ YES" if identity_match else "❌ NO"}')
    print(f'  Authentication  : {"🔓 GRANTED" if auth_pass else "🔒 DENIED"}')
    print(f'{'='*55}')
    return auth_pass, predicted_id, confidence


print('📡 RF-Based Device Authentication Simulation')
print('Authenticating all devices with legitimate signals:')
for dev_id in range(NUM_DEVICES):
    authenticate_device(dev_id, confidence_threshold=0.75)

print('\n🚨 Spoofing Attack Simulation:')
print('Device 3 tries to claim it is Device 1 (attacker uses wrong device)')
# Attacker generates signal from device 3 but claims to be device 1
I, Q = generate_rf_signal(device_id=3)   # attacker's real device
spec  = iq_to_spectrogram(I, Q)
x     = torch.from_numpy(spec[np.newaxis, np.newaxis, :, :].astype(np.float32)).to(device)
model.eval()
with torch.no_grad():
    probs = F.softmax(model(x), dim=1)[0].cpu().numpy()
predicted_id  = probs.argmax()
confidence    = probs[predicted_id]
claimed_id    = 0  # attacker claims device 1
print(f'\nClaimed: Device 1 | RF fingerprint matches Device {predicted_id+1} | '
      f'Auth: {"DENIED 🔒" if predicted_id != claimed_id else "GRANTED 🔓"}')

## 📊 Step 11: Per-Device Accuracy & Confidence Analysis

In [ ]:
import torch.nn.functional as F

model.eval()
all_probs = []
with torch.no_grad():
    for Xb, yb in test_loader:
        probs = F.softmax(model(Xb.to(device)), dim=1).cpu().numpy()
        all_probs.append(probs)
all_probs = np.vstack(all_probs)

per_class_acc  = []
per_class_conf = []
for dev_id in range(NUM_DEVICES):
    mask = all_labels == dev_id
    acc  = (all_preds[mask] == dev_id).mean() * 100
    conf = all_probs[mask, dev_id].mean() * 100
    per_class_acc.append(acc)
    per_class_conf.append(conf)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(NUM_DEVICES)
w = 0.35
bars1 = ax.bar(x - w/2, per_class_acc,  w, label='Accuracy (%)',   color='steelblue',  alpha=0.85)
bars2 = ax.bar(x + w/2, per_class_conf, w, label='Avg Confidence (%)', color='tomato', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(device_names, rotation=15)
ax.set_ylim(0, 115); ax.set_ylabel('Percentage (%)'); ax.set_title('Per-Device Accuracy & Confidence')
ax.legend(); ax.grid(axis='y', alpha=0.3)
for bar in bars1: ax.annotate(f'{bar.get_height():.1f}', xy=(bar.get_x()+bar.get_width()/2, bar.get_height()+1), ha='center', va='bottom', fontsize=8)
for bar in bars2: ax.annotate(f'{bar.get_height():.1f}', xy=(bar.get_x()+bar.get_width()/2, bar.get_height()+1), ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.savefig('per_device_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

## ✅ Summary

| Component | Detail |
|-----------|--------|
| **Dataset** | Simulated I/Q RF signals with hardware impairments (CFO, phase noise, I/Q imbalance) |
| **Feature** | 64×64 power spectrograms (dB scale, normalized) |
| **Model** | 3-block CNN with BatchNorm + Dropout |
| **Task** | Multi-class classification (device identification) |
| **Security** | RF fingerprinting detects spoofing attacks |

**Next Steps:**
- Use a **real RF dataset** (e.g., [DeepSig RadioML](https://www.deepsig.ai/datasets) or [ORACLE](https://oracle.wns.io/))
- Try **ResNet / EfficientNet** backbones for higher accuracy
- Add **few-shot learning** for new/unknown devices
- Deploy as a **real-time authentication API**